# Current Trends in LLMs — (with NRP OpenAI-compatible API)

*Last refreshed: June 2026.*

Every demo here makes API calls to a model through the OpenAI-compatible NRP endpoint using `chat.completions`. The code is intentionally kept short but illustrative.

### Available models on the endpoint
| Model | Status | Params | Context | Tools | Reason | Inputs |
|---|---|---|---|---|---|---|
| `qwen3` | main | 397B | 1.01M | ✓ | ✓ | image, video |
| `qwen3-small` | main | 27B | 1.01M | ✓ | ✓ | image, video |
| `gpt-oss` | main | 120B | 131K | ✓ | ✓ | — |
| `gemma` | main | 31B | 262K | ✓ | ✓ | image, video |
| `gemma-small` | evaluating | ~8B | 131K | ✓ | ✓ | image, video, audio |
| `kimi` | evaluating | 1T | 262K | ✓ | ✓ | image, video |
| `glm-5` | evaluating | 744B | 203K | ✓ | ✓ | — |
| `minimax-m2` | evaluating | 230B | 205K | ✓ | ✓ | — |
| `qwen3-embedding` | main | 8B | — | — | — | image, video |

This notebook defaults to the models `qwen3-small` (fast tier), `qwen3` (frontier / vision / tools), and `qwen3-embedding`. Swap in any model name above to compare.

### Caveats
- **Insert your API key** — running needs *your* key. Never paste a key into a shared notebook.
- **Results vary run to run.**  Including dependence on the gateway load: https://nrp.ai/llm-status/
- **Reasoning surfacing is gateway-specific.** Cell 3 reads the `reasoning` field, but this can vary across providers/gateways.
- **Not here:** scaling laws and facts about *training* -- these are not observable from single API calls.


In [ ]:
import os, time, json, re, base64, io
from collections import Counter
import numpy as np
from openai import OpenAI

# This assumes that you have a "keys.py" file in this directory
# with NRP_TOK assigned the value of your NRP API token (required)
# and NRP_CACHE_SALT assigned your cache_salt value (optional)
import keys
NRP_TOK = keys.NRP_TOK
NRP_CACHE_SALT = keys.NRP_CACHE_SALT

llm_client = OpenAI(api_key = NRP_TOK,
                    base_url = "https://ellm.nrp-nautilus.io/v1")

# The "main" models above.
FAST_MODEL     = "qwen3-small"   # 27B
FRONTIER_MODEL = "qwen3"         # 397B
EMBED_MODEL    = "qwen3-embedding"

def content_of(completion):
    """First choice's text content (empty string if none)."""
    return completion.choices[0].message.content or ""

## Model tiers — the fast-vs-frontier split

Practically speaking, we will always be interested in how quickly we can get a response from our LLM as well as how good the response is.  

From an efficiency stand-point, we can route easy work to a small fast model and hard work to a big one. Below we send the same prompt to both models and watch the trade-off in latency and output length. (Try replacing with other models to extend the comparison.)

In [ ]:
prompt = "In one sentence, what is a transformer in machine learning?"

for model in [FAST_MODEL, FRONTIER_MODEL]:
    t0 = time.time()
    completion = llm_client.chat.completions.create(
        model=model, 
        #max_tokens=200,
        messages=[{"role": "user", "content": prompt}],
        extra_body={"cache_salt": NRP_CACHE_SALT}
    )
    dt = time.time() - t0
    prompt_tok = completion.usage.prompt_tokens if completion.usage else "?"
    used_tok = completion.usage.completion_tokens if completion.usage else "?"
    total_tok = completion.usage.total_tokens if completion.usage else "?"
    print(f"\n{model}")
    print(f"  latency: {dt:.1f}s")
    print(f"  prompt tokens: {prompt_tok}")
    print(f"  completion tokens: {used_tok}")
    print(f"  total tokens: {total_tok}")
    print(" ", content_of(completion))

In [ ]:
completion

## Inference-time compute — self-consistency

"Think longer" in its simplest form: ask the same question several times at high temperature and take a majority vote. Individual samples may disagree; the aggregate is steadier than any single attempt. The cost scales with the number of samples — this is a trade-off in inference-time-compute.

In [ ]:
question = (
    "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. "
    "How much does the ball cost, in cents? "
    "Reason briefly, then end with a line exactly like 'ANSWER: <number>'."
)

def sample_answer():
    completion = llm_client.chat.completions.create(
        model=FAST_MODEL, 
        #max_tokens=400, 
        temperature=1.0,
        messages=[{"role": "user", 
                   "content": question}],
        extra_body={"cache_salt": NRP_CACHE_SALT,
                    "chat_template_kwargs": {"enable_thinking": False}}
    )
    found = re.findall(r"ANSWER:\s*\$?(\d+)", content_of(completion))
    return int(found[-1]) if found else None

In [ ]:
sample_answer()

In [ ]:
votes = [sample_answer() for _ in range(9)]
print("individual answers:", votes)

tally = Counter(v for v in votes if v is not None)
if tally:
    winner, n = tally.most_common(1)[0]
    print(f"majority vote: {winner} cents  ({n}/{len(votes)} agreed)")

## Reasoning models — surfacing the thinking

These models reason before answering.

Enabling or deepening reasoning is gateway-specific — e.g. some accept `reasoning_effort="high"` or `extra_body={"chat_template_kwargs": {"enable_thinking": True}}`.

In [ ]:
completion = llm_client.chat.completions.create(
    model=FAST_MODEL, 
    max_tokens=3000,
    messages=[{"role": "user",
               "content": "Explain why the sum of any two odd numbers is always even."}],
    extra_body={"cache_salt": NRP_CACHE_SALT}
)
message = completion.choices[0].message
reasoning = message.reasoning
content = message.content

if reasoning:
    print("--- REASONING (first 400 chars) ---")
    print(reasoning[:400], "...\n")
else:
    print("(we did not get a separate reasoning trace)\n")

print("--- ANSWER ---")
print(content)

## Multimodality — sending a real image

We draw a picture locally with PIL, encode it as a base64 data URL, and pass it as an `image_url` content part — the OpenAI-compatible way to do vision. No external files: the model reasons over pixels we just generated.

In [ ]:
from PIL import Image, ImageDraw

img = Image.new("RGB", (200, 200), "white")
draw = ImageDraw.Draw(img)
draw.ellipse([40, 40, 160, 160], fill="red")
draw.rectangle([85, 85, 115, 115], fill="blue")

buf = io.BytesIO()
img.save(buf, format="PNG")
data_url = "data:image/png;base64," + base64.standard_b64encode(buf.getvalue()).decode()
img.save("output.png")

completion = llm_client.chat.completions.create(
    model=FAST_MODEL, 
    #max_tokens=200,
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "What shapes and colors do you see in this image?"},
            {"type": "image_url", "image_url": {"url": data_url}},
        ],
    }],
    extra_body={"cache_salt": NRP_CACHE_SALT}
)
print(content_of(completion))

## Agentic systems — the tool-using loop

We give the model one tool (a calculator), then loop: the model emits a `tool_calls` request, we actually run the Python function, return the result as a `role: "tool"` message, and continue until it answers. This is an agent loop in OpenAI-compatible form.

Every extra tool step is another place the chain can break -- end-to-end reliability decays faster than per-step reliability.  For agentic systems, be watchful of how many calls a multi-step problem needs.

In [ ]:
def calculator(expression):
    """Evaluate a basic arithmetic expression. (Toy tool — restricted builtins.)"""
    return str(eval(expression, {"__builtins__": {}}, {}))

tools = [{
    "type": "function",
    "function": {
        "name": "calculator",
        "description": "Evaluate a Python arithmetic expression, e.g. '17 * 4.5'.",
        "parameters": {
            "type": "object",
            "properties": {"expression": {"type": "string"}},
            "required": ["expression"],
        },
    },
}]

def run_agent(task, max_turns=5):
    messages = [{"role": "user", "content": task}]
    for _ in range(max_turns):
        completion = llm_client.chat.completions.create(
            model=FAST_MODEL, 
            max_tokens=1000, 
            tools=tools, 
            messages=messages,
            extra_body={"cache_salt": NRP_CACHE_SALT}
        )
        message = completion.choices[0].message
        messages.append(message.model_dump())          # keep full assistant turn
        if not message.tool_calls:                      # no tool -> final answer
            return message.content
        for call in message.tool_calls:
            args = json.loads(call.function.arguments)
            out = calculator(**args)
            print(f"  [tool] calculator({args['expression']}) = {out}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": out})
    return "(stopped: hit max turns)"

answer = run_agent(
    "A shop sells pens at 3 for $4.50. If I buy 17 pens, what do I pay in total? "
    "Use the calculator tool for any arithmetic."
)
print("\nFINAL:", answer)

## Structured output — JSON you can parse

Production systems want fields, not prose. Ask for strict JSON and parse it. The cell tolerates accidental code fences; many gateways also support `response_format={"type": "json_object"}`, which you can add for a hard guarantee.

In [ ]:
prompt = (
    "Extract the following sentence into JSON with keys 'name', 'role', and 'company'. "
    "Return ONLY the JSON object, no prose. "
    "Sentence: 'Dario Amodei is the chief executive of Anthropic.'"
)
completion = llm_client.chat.completions.create(
    model=FAST_MODEL, 
    #max_tokens=200,
    messages=[{"role": "user", "content": prompt}],
    extra_body={"cache_salt": NRP_CACHE_SALT},
    response_format={"type": "json_object"},
)

In [ ]:
content_of(completion)

In [ ]:
json.loads(content_of(completion))

In [ ]:
raw = content_of(completion).strip().strip("`")
raw = re.sub(r"^json", "", raw).strip()
data = json.loads(raw)
print(type(data).__name__, "->", data)
print("company field:", data["company"])

## Embeddings — our representation space

Here we compute real embeddings with `qwen3-embedding` and look at the cosine-similarity matrix. Semantically similar sentences should score high; unrelated ones low. This is the representation that powers retrieval (RAG) and semantic search.

In [ ]:
sentences = [
    "A cat sat on the mat.",
    "A kitten rested on the rug.",
    "The central bank raised interest rates today.",
]
resp = llm_client.embeddings.create(model=EMBED_MODEL, 
                                    input=sentences,
                                    extra_body={"cache_salt": NRP_CACHE_SALT})
vecs = np.array([d.embedding for d in resp.data])

unit = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)
sim = unit @ unit.T

print("cosine similarity matrix:")
for i in range(len(sentences)):
    print("  ", " ".join(f"{sim[i, j]:.2f}" for j in range(len(sentences))))
print("\nrow 0 vs 1 (cat/kitten):", f"{sim[0,1]:.2f}",
      " | row 0 vs 2 (cat/rates):", f"{sim[0,2]:.2f}")

## Measuring the cost of scale

This is the empirical complement to scaling-law plots: we send the same prompt to several models spanning the size range on the endpoint, and measure two things against parameter count — latency (wall-clock per call) and throughput (output tokens per second).

Throughput is the fairer cost measure: latency partly reflects how much each model chose to generate, while tokens/sec normalizes for that. Note it's end-to-end throughput — it includes queueing and prompt processing, not just raw decode speed — so it's a practical cost number, not a hardware benchmark.

Each model runs a few times and we average to smooth jitter, and the loop is wrapped so a slow or unavailable model just gets skipped rather than halting the sweep. Both numbers depend on serving setup, batching, and load — not only parameter count — so read the trend, and expect noise.

In [ ]:
import matplotlib.pyplot as plt

# (model name, parameter count in billions) — from the endpoint's table.
sweep = [
    ("qwen3-small", 27),
    ("gemma",       31),
    ("gpt-oss",     120),
    ("minimax-m2",  230),
    ("qwen3",       397),
    ("glm-5",       744), 
    ("kimi",        1000), 
]
prompt = "Name three uses of large language models. Answer in one short sentence."
RUNS = 3          # average over a few calls per model to smooth jitter

results = []      # (name, params, latency_s, tokens_per_s or None)
for name, params in sweep:
    times, toks = [], []
    try:
        for _ in range(RUNS):
            t0 = time.time()
            completion = llm_client.chat.completions.create(
                model=name, 
                # max_tokens=80,
                messages=[{"role": "user", "content": prompt}],
                extra_body={"cache_salt": NRP_CACHE_SALT}
            )
            times.append(time.time() - t0)
            if completion.usage:
                toks.append(completion.usage.completion_tokens)
        latency = sum(times) / len(times)
        tps = (sum(toks) / sum(times)) if toks else None   # end-to-end output throughput
        results.append((name, params, latency, tps))
        tps_str = f"{tps:5.1f} tok/s" if tps else "   n/a    "
        print(f"{name:13s} {params:>4d}B  ->  {latency:5.2f}s avg | {tps_str}")
    except Exception as e:
        print(f"{name:13s} skipped ({type(e).__name__}: {e})")

if results:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # left: latency vs size (lower is better)
    for name, params, latency, tps in results:
        ax1.scatter(params, latency, s=60)
        ax1.annotate(name, (params, latency), textcoords="offset points", xytext=(6, 5))
    ax1.set_xscale("log")
    ax1.set_xlabel("parameters (billions, log scale)")
    ax1.set_ylabel("avg latency (seconds)")
    ax1.set_title("Latency vs model size  (lower = better)")
    ax1.grid(True, which="both", alpha=0.3)

    # right: throughput vs size (higher is better) — only models that reported usage
    tps_points = [(params, tps, name) for name, params, latency, tps in results if tps]
    if tps_points:
        for params, tps, name in tps_points:
            ax2.scatter(params, tps, s=60, color="tab:green")
            ax2.annotate(name, (params, tps), textcoords="offset points", xytext=(6, 5))
        ax2.set_xscale("log")
        ax2.set_xlabel("parameters (billions, log scale)")
        ax2.set_ylabel("output tokens / second")
        ax2.set_title("Throughput vs model size  (higher = better)")
        ax2.grid(True, which="both", alpha=0.3)
    else:
        ax2.text(0.5, 0.5, "no token usage reported\nby this endpoint",
                 ha="center", va="center")
        ax2.set_axis_off()

    plt.tight_layout()
    plt.show()

The below restricts the model calls with `max_tokens=80`:

In [ ]:
import matplotlib.pyplot as plt

# (model name, parameter count in billions) — from the endpoint's table.
sweep = [
    ("qwen3-small", 27),
    ("gemma",       31),
    ("gpt-oss",     120),
    ("minimax-m2",  230),
    ("qwen3",       397),
    ("glm-5",       744), 
    ("kimi",        1000), 
]
prompt = "Name three uses of large language models. Answer in one short sentence."
RUNS = 3          # average over a few calls per model to smooth jitter

results = []      # (name, params, latency_s, tokens_per_s or None)
for name, params in sweep:
    times, toks = [], []
    try:
        for _ in range(RUNS):
            t0 = time.time()
            completion = llm_client.chat.completions.create(
                model=name, 
                max_tokens=80,
                messages=[{"role": "user", "content": prompt}],
                extra_body={"cache_salt": NRP_CACHE_SALT}
            )
            times.append(time.time() - t0)
            if completion.usage:
                toks.append(completion.usage.completion_tokens)
        latency = sum(times) / len(times)
        tps = (sum(toks) / sum(times)) if toks else None   # end-to-end output throughput
        results.append((name, params, latency, tps))
        tps_str = f"{tps:5.1f} tok/s" if tps else "   n/a    "
        print(f"{name:13s} {params:>4d}B  ->  {latency:5.2f}s avg | {tps_str}")
    except Exception as e:
        print(f"{name:13s} skipped ({type(e).__name__}: {e})")

if results:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # left: latency vs size (lower is better)
    for name, params, latency, tps in results:
        ax1.scatter(params, latency, s=60)
        ax1.annotate(name, (params, latency), textcoords="offset points", xytext=(6, 5))
    ax1.set_xscale("log")
    ax1.set_xlabel("parameters (billions, log scale)")
    ax1.set_ylabel("avg latency (seconds)")
    ax1.set_title("Latency vs model size  (lower = better)")
    ax1.grid(True, which="both", alpha=0.3)

    # right: throughput vs size (higher is better) — only models that reported usage
    tps_points = [(params, tps, name) for name, params, latency, tps in results if tps]
    if tps_points:
        for params, tps, name in tps_points:
            ax2.scatter(params, tps, s=60, color="tab:green")
            ax2.annotate(name, (params, tps), textcoords="offset points", xytext=(6, 5))
        ax2.set_xscale("log")
        ax2.set_xlabel("parameters (billions, log scale)")
        ax2.set_ylabel("output tokens / second")
        ax2.set_title("Throughput vs model size  (higher = better)")
        ax2.grid(True, which="both", alpha=0.3)
    else:
        ax2.text(0.5, 0.5, "no token usage reported\nby this endpoint",
                 ha="center", va="center")
        ax2.set_axis_off()

    plt.tight_layout()
    plt.show()

## Notes

Each cell maps onto a trend, were we are running against real open-weight models:

- **Model tiers** — the efficiency frontier as a routing decision; the smaller model is faster, the larger more thorough.
- **Self-consistency** — the cheapest inference-time compute, with its built-in limit: aggregation only helps when one attempt already beats chance.
- **Reasoning** — explicit deliberation before the answer.
- **Vision** — native multimodality over an image you generated.
- **Tool-using agent** — the act/observe loop; potential error-compounding will bite.
- **Structured output** — how models get wired into software.
- **Embeddings** — the representation space behind semantic search and RAG.
- **Latency & throughput vs size** — the cost side of scaling, measured against our endpoint: wall-clock latency and output tokens/second across the model range.